In [ ]:
# ==========================================================
# Compare RNN, CNN, and LSTM Models
# Metrics: Accuracy, F1-score, Loss, Confusion Matrix, Error Rate
# Dataset: IMDB Sentiment Classification
# ==========================================================

import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# -----------------------------
# 1. Load Dataset
# -----------------------------
vocab_size = 10000
max_len = 200

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

# -----------------------------
# 2. Build Models
# -----------------------------

def build_rnn():
    model = Sequential([
        Embedding(vocab_size, 128, input_length=max_len),
        SimpleRNN(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


def build_lstm():
    model = Sequential([
        Embedding(vocab_size, 128, input_length=max_len),
        LSTM(64),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


def build_cnn():
    model = Sequential([
        Embedding(vocab_size, 128, input_length=max_len),
        Conv1D(filters=128, kernel_size=5, activation='relu'),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model


models = {
    "RNN": build_rnn(),
    "LSTM": build_lstm(),
    "CNN": build_cnn()
}

# -----------------------------
# 3. Train and Evaluate Models
# -----------------------------

results = {}

for name, model in models.items():
    print(f"\nTraining {name} model...")

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train, y_train,
        epochs=3,
        batch_size=64,
        validation_split=0.2,
        verbose=1
    )

    # Prediction
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype("int32")

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    loss, keras_accuracy = model.evaluate(X_test, y_test, verbose=0)
    error_rate = 1 - accuracy
    cm = confusion_matrix(y_test, y_pred)

    results[name] = {
        "Accuracy": accuracy,
        "F1-score": f1,
        "Loss": loss,
        "Error Rate": error_rate,
        "Confusion Matrix": cm
    }

    print(f"\n{name} Results")
    print("Accuracy:", accuracy)
    print("F1-score:", f1)
    print("Loss:", loss)
    print("Error Rate:", error_rate)
    print("Confusion Matrix:\n", cm)

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# -----------------------------
# 4. Display Final Comparison
# -----------------------------

print("\nFinal Comparison")
print("======================================")

for model_name, metrics in results.items():
    print(f"\n{model_name}")
    print(f"Accuracy   : {metrics['Accuracy']:.4f}")
    print(f"F1-score   : {metrics['F1-score']:.4f}")
    print(f"Loss       : {metrics['Loss']:.4f}")
    print(f"Error Rate : {metrics['Error Rate']:.4f}")

# -----------------------------
# 5. Plot Confusion Matrices
# -----------------------------

for model_name, metrics in results.items():
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        metrics["Confusion Matrix"],
        annot=True,
        fmt="d",
        cmap="Blues"
    )
    plt.title(f"{model_name} Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.show()

In [ ]:
import pandas as pd

comparison_table = pd.DataFrame({
    model: {
        "Accuracy": results[model]["Accuracy"],
        "F1-score": results[model]["F1-score"],
        "Loss": results[model]["Loss"],
        "Error Rate": results[model]["Error Rate"]
    }
    for model in results
}).T

print(comparison_table)

In [ ]:
print(X_train[0])

In [ ]:
print(y_train[0])

In [ ]:
from tensorflow.keras.datasets import imdb

# Load dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data()

# Load word dictionary
word_index = imdb.get_word_index()

# Reverse dictionary: numbers -> words
reverse_word_index = {
    value: key for (key, value) in word_index.items()
}

# Print meaning of number 14
print(reverse_word_index[14])
print(reverse_word_index[15])

In [ ]:
from tensorflow.keras.datasets import imdb

(X_train, y_train), (X_test, y_test) = imdb.load_data()

word_index = imdb.get_word_index()

reverse_word_index = {
    value: key for (key, value) in word_index.items()
}

decoded_review = " ".join(
    reverse_word_index.get(i - 3, "?")
    for i in X_train[0]
)

print(decoded_review)